# Project: European Air Traffic Pipeline on EUROCONTROL OPDI data (Medallion Architecture, PySpark)

**DAT535 - Data Engineering**

Thomas Canisares--Morère & Mathis Nancey - Group 08

In [ ]:
# Setup and Imports
import os
import json
import time
from datetime import datetime, timedelta
import random
import csv
import hashlib
import shutil
import urllib.error
import urllib.request
from pathlib import Path
from collections import Counter


import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

print("Libraries imported successfully")


In [ ]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Project_Thomas_Mathis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


## Part 1: Retrieve the data

Download of OPDI raw data (EUROCONTROL)

- Period: April 1, 2026 -> July 28, 2026 (~10.3 GB)
- Tables: flight_list (monthly), flight_events and measurements (10-day intervals), OurAirports repositories (airports, runways)

In [ ]:
# Parameters
START, END = datetime.date(2026, 4, 1), datetime.date(2026, 7, 29)
ORIGIN     = datetime.date(2022, 1, 1)   # the 10-day periods are counted starting from this date
MIN_FREE_GB = 4.0     # minimum free space in GB before downloading a file
RETRIES     = 3       # number of download attempts before giving up
PAUSE_S     = 0.5     # pause between downloads in seconds
RAW_DIR    = Path.home() / "opdi-data" / "raw"

UA = {"User-Agent": "UiS-DAT535-group08"} # user agent for HTTP requests
BASE_URL   = "https://www.eurocontrol.int/performance/data/download/OPDI/v002"
REF_URLS   = {"airports.csv": "https://davidmegginson.github.io/ourairports-data/airports.csv",
              "runways.csv":  "https://davidmegginson.github.io/ourairports-data/runways.csv"}

# List of files to retrieve
# months() generates the first day of each month in [start, end[
def months(start, end):
    m = datetime.date(start.year, start.month, 1)
    while m < end:
        yield m
        m = datetime.date(m.year + (m.month == 12), m.month % 12 + 1, 1)

# Generate 10-day windows starting from ORIGIN
def windows(start, end, days=10):
    w = ORIGIN + datetime.timedelta(days=((start - ORIGIN).days // days) * days)
    while w < end:
        yield w, w + datetime.timedelta(days=days)
        w += datetime.timedelta(days=days)

# We add the flight_list files for each month, and the flight_events and measurements files for each 10-day window
files = []
for m in months(START, END):
    n = f"flight_list_{m:%Y%m}.parquet"
    files.append(("flight_list", n, f"{BASE_URL}/flight_list/{n}"))
for table in ("flight_events", "measurements"):
    for w, w_end in windows(START, END):
        n = f"{table}_{w:%Y%m%d}_{w_end:%Y%m%d}.parquet"
        files.append((table, n, f"{BASE_URL}/{table}/{n}"))
for n, url in REF_URLS.items():
    files.append(("reference", n, url))

# We return here the HTTP status, the expected size in bytes, and the last modified date (if available)
def head(url):
    req = urllib.request.Request(url, method="HEAD", headers=UA)
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status, int(r.headers.get("Content-Length") or 0), r.headers.get("Last-Modified", "")
    except urllib.error.HTTPError as e:
        return e.code, 0, ""
    except Exception as e:
        return type(e).__name__, 0, ""

# this function downloads the file from the given URL to the destination path, 
# checking that the size matches the expected size. 
# It retries up to RETRIES times in case of failure.
#    - url: the URL to download from
#    - dest: the destination path (Path object)
def download(url, dest):
    tmp = dest.with_name(dest.name + ".part") # temporary file during download
    for attempt in range(1, RETRIES + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20) # copy in 1 MB chunks
            os.replace(tmp, dest)
            return "downloaded"
        except Exception as e:
            print(f"    tentative {attempt}/{RETRIES} échouée : {e}")
            tmp.unlink(missing_ok=True)
            time.sleep(5 * attempt)
    return "failed"

# Compute the SHA256 hash of a file at the given path
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Return the free disk space in GB for the RAW_DIR
def free_gb():
    return shutil.disk_usage(RAW_DIR).free / 1e9

# Return the current UTC time as a datetime object 
def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)

# The main loop that downloads the files and records the results in a manifest CSV file.
# The manifest contains all the relevant information about each file downloaded.
# It contains the following columns:
# run_id, table, file, url, http_status, size_bytes, last_modified, result, sha256, local_path, retrieved_at_utc
#  - run_id: a unique identifier for this run of the script
#  - table: the table name (flight_list, flight_events, measurements, reference)
#  - file: the file name
#  - url: the URL from which the file was downloaded
#  - http_status: the HTTP status code returned by the server
#  - size_bytes: the size of the file in bytes
#  - last_modified: the last modified date of the file on the server (if available)
#  - result: the result of the download (downloaded, already_present, absent, empty, failed)
#  - sha256: the SHA256 hash of the file
#  - local_path: the local path of the downloaded file
#  - retrieved_at_utc: the UTC time when the file was retrieved
RAW_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = RAW_DIR / "manifest.csv"
run_id = utc_now().strftime("%Y%m%dT%H%M%SZ")
print(f"Période [{START}, {END}[ | {len(files)} fichiers | destination {RAW_DIR} | libre {free_gb():.1f} Go\n")
rows, t0 = [], time.time() # list to store the results and the start time
for i, (table, name, url) in enumerate(files, 1):
    status, size, last_modified = head(url)
    dest = RAW_DIR / table / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if status != 200:
        result = "absent"
    elif size == 0:
        result = "empty"
    elif dest.exists():
        result = "already_present"
    elif free_gb() - size / 1e9 < MIN_FREE_GB:
        print(f"ARRÊT : espace disque insuffisant avant {name}")
        break
    else:
        result = download(url, dest)
    ok = result in ("downloaded", "already_present")
    rows.append(dict(run_id=run_id, table=table, file=name, url=url, http_status=status,
                     size_bytes=size, last_modified=last_modified, result=result,
                     sha256=sha256(dest) if ok else "", local_path=str(dest) if ok else "",
                     retrieved_at_utc=utc_now().isoformat(timespec="seconds")))
    print(f"[{i:2d}/{len(files)}] {table:13s} {name:45s} {size/1e6:8.1f} Mo  {result}")
    time.sleep(PAUSE_S)

if rows: # write the manifest CSV file
    new = not manifest_path.exists()
    with open(manifest_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        if new:
            w.writeheader()
        w.writerows(rows)

print(f"\nRésultats : {dict(Counter(r['result'] for r in rows))}") # print summary of results
for t in ("flight_list", "flight_events", "measurements", "reference"): # print summary of files downloaded for each table
    sel = [r for r in rows if r["table"] == t and r["local_path"]]
    print(f"  {t:13s} {len(sel):3d} fichiers  {sum(int(r['size_bytes']) for r in sel)/1e9:6.2f} Go")
print(f"Durée {time.time()-t0:.0f} s | espace libre restant {free_gb():.1f} Go | manifeste : {manifest_path}") # duration, free space, manifest path
for r in rows: # print any files that were not downloaded successfully
    if r["result"] not in ("downloaded", "already_present"):
        print(f"  ! {r['file']} -> {r['result']} (HTTP {r['http_status']})")